<table><tr>
<td><b>Sapienza University of Rome</b><br>PhD Soft Skills 2026</td>
</tr></table>

# Notebook 1 — Teach a machine to sort your texts
**30 minutes.** You will build, train, evaluate and then deliberately break a working machine-learning system.


---
### How to use this notebook

1. Each numbered section gives you **a prompt**. Copy it.
2. Find the **empty code cell** underneath it.
3. **Click the word `generate`** inside that cell — it reads *Start coding or generate with AI*. An AI box opens. Paste the prompt there and press Enter.
4. **Look at what it wrote** — even if you don't understand it — then press ▶.
5. If it fails, don't fix it by hand. Paste the whole red error message back to the AI.

Every section also has a **Reference** cell with working code already in it. If you would rather watch than type, just run that instead. You will lose nothing.

> **Before you start: click `Copy to Drive` in the toolbar.** This notebook opened straight from GitHub, which means it is read-only — you can run it, but nothing you do will be saved. One click makes it yours to keep.

> The first time you run a cell you may see **"This notebook was not authored by Google"**. That is normal for anything opened from GitHub. Click **Run anyway**. (Copying to Drive first usually avoids it altogether.)

> You are never expected to write code. You are expected to say clearly what you want, and to check what you get.
---

## About the corpus

`letters.csv` holds 800 short letters in the style of a 17th-century Italian correspondence archive. Each is labelled **petition** (the writer is asking for something) or **report** (the writer is informing).

**It is constructed, not historical.** It was written for this course so that it behaves like a real collection in the ways that matter: the two classes overlap, some letters are genuinely ambiguous, and **exactly one label in eight is wrong** — as in any hand-catalogued archive.

That sets a ceiling: with 12% of labels wrong, neither a model nor a human could score much above the high 80s. Anything in the **70s or low 80s is a good result here** — and your own number will differ from the reference by a few points, because your split of the data is different.

That last point matters. It means you cannot reach 100%, and neither could a human.

**Two things you will see and can ignore for now.** The setup cell also unpacks `dates_gold.csv` and an `archive/` folder — those belong to later notebooks. And the table has two extra columns: `true_label` (the label before we added realistic noise — using it would be cheating) and `iso_date` (used in Notebook 5). Work only with `text` and `label`.

In [ ]:
#@title Setup — click the play button on the left. Nothing is installed.
# The long string below is the course corpus, packed into this notebook.
# You never need to read it.
#
# The exercise prompts each begin by downloading the same corpus from
# the course website. That is deliberate: it makes every prompt work on
# its own, wherever you paste it. This cell is the safety net, so the
# exercises still run if the site is unreachable. Same files either way.
import base64, gzip, io, tarfile, pathlib

_BLOB = (
    "..."
    "uOOOO+7I8/8DgI/8jQBwCAA="
)

_tar = tarfile.open(fileobj=io.BytesIO(gzip.decompress(base64.b64decode(_BLOB))))
try:
    _tar.extractall('.', filter='data')   # Python 3.12+
except TypeError:
    _tar.extractall('.')                  # older Python
print('Ready. Files now available:')
for p in sorted(pathlib.Path('.').glob('*.csv')):
    print('  ', p, f'({p.stat().st_size//1024} KB)')
for _dir, _ext, _what in [('archive', '*.txt', 'text files'), ('pdfs', '*.pdf', 'PDFs'),
                          ('images', '*.png', 'images'),
                          ('images_clean', '*.png', 'images (the other survey)')]:
    _n = len(list(pathlib.Path(_dir).glob(_ext))) if pathlib.Path(_dir).is_dir() else 0
    if _n:                      # not every notebook needs every folder
        print(f'   {_dir + "/":9s}:', _n, _what)
print()
print('These same files live at')
print('https://fabsilvestri.github.io/ai-from-the-inside/corpus.zip')
print('The exercise prompts fetch them from there, so each prompt stands alone.')

### 1 · Look at your data before anything else

**Copy this into the AI box** — click the word **generate** in the empty cell below, then paste this in:

```text
Download https://fabsilvestri.github.io/ai-from-the-inside/corpus.zip
and unzip it into the current folder, so that letters.csv and the
archive/ and pdfs/ folders are alongside this notebook.

Load the file letters.csv into a table.
Show me its shape, the column names, and 10 random rows
with the full text visible.
Also show how many letters carry each label.
```

> Never train anything before you have looked at the data. You are checking: is it what I think it is?

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)   # show letters in full

df = pd.read_csv('letters.csv')
print('shape:', df.shape)
print('columns:', list(df.columns))
print()
print(df['label'].value_counts())
print()
display(df.sample(10, random_state=1)[['id', 'text', 'label']])

### 2 · Train and evaluate — the whole of machine learning, in one prompt

**Copy this into the AI box** — click the word **generate** in the empty cell below, then paste this in:

```text
Download https://fabsilvestri.github.io/ai-from-the-inside/corpus.zip
and unzip it into the current folder, so that letters.csv and the
archive/ and pdfs/ folders are alongside this notebook.

Using the letters.csv data, split it 80/20 into a training set and a test
set, keeping the balance of labels the same in both.
Train a TF-IDF + logistic regression text classifier on the training set
only, and call the fitted model `model`.
Report its accuracy on the test set and store it in a variable called `acc`.
Use the 'label' column as the answer, not 'true_label'.
```

> Note the last line. The file also contains `true_label`, which is the label *before* we introduced realistic noise. Using it would be cheating — and it is exactly the kind of leak that ruins real studies.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=4, stratify=df['label'])

model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2)),
    LogisticRegression(max_iter=2000))
model.fit(X_train, y_train)

acc = accuracy_score(y_test, model.predict(X_test))
print(f'Test accuracy: {acc:.1%}   ({len(X_train)} training, {len(X_test)} test)')

### 3 · The baseline — the number nobody reports

**Copy this into the AI box** — click the word **generate** in the empty cell below, then paste this in:

```text
What accuracy would I get on the same test set by ignoring the text
completely and always predicting the most common label?
Print that number next to the model's accuracy so I can compare them.
```

> **If these two numbers are close, your model has learned nothing.** Ask this of every accuracy figure you ever meet, including in papers you referee.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
base = accuracy_score(y_test, dummy.predict(X_test))

print(f'Do-nothing baseline : {base:.1%}')
print(f'Our model           : {acc:.1%}')
print(f'Actually learned    : {acc - base:+.1%}')

### 4 · Read the mistakes

**Copy this into the AI box** — click the word **generate** in the empty cell below, then paste this in:

```text
Show me 8 test letters the model got wrong.
For each one show the full text, the label recorded in the 'label' column,
and the label the model predicted, in a readable layout — not a cramped table.
Do not use the 'true_label' column.
```

> Read them properly. Ask yourself: *would I have got this right?* Some of them are mislabelled, and some are genuinely ambiguous. This is the most valuable five minutes in the notebook.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
import textwrap

pred = model.predict(X_test)
wrong = [(t, a, b) for t, a, b in zip(X_test, y_test, pred) if a != b]

print(f'{len(wrong)} mistakes out of {len(X_test)} test letters\n')
print('(showing the first 8)\n')
for text, recorded, guess in wrong[:8]:
    print(f'RECORDED: {recorded:9s} MODEL SAID: {guess}')
    print(textwrap.fill(text, 92))
    print('-' * 92)

### 5 · What did it actually learn?

**Copy this into the AI box** — click the word **generate** in the empty cell below, then paste this in:

```text
Show me the 15 words or phrases that push the model most strongly towards
each of the two labels. Display them as a horizontal bar chart, with the
two labels in different colours.
```

> Look at the two sides separately, because they tell different stories.

Towards **petition** you should see honest content words — `ask`, `request`, `beg`, `ask that`. That is the model finding the actual signal, and it is reassuring.

Towards **report** you will see something else entirely: fragments of the *closing formula* — `kiss your hands`, `remain wholly`, `at your disposal` — and very possibly a **place name** such as `bologna`. Neither has the faintest thing to do with whether a letter asks for something.

That is a **shortcut**, and it is the same failure as the skin-lesion classifier that learned to detect rulers. The writers who used the most elaborate farewell happened to be reporting rather than asking, and the model seized on it. It works here. It would collapse on any archive with different scribal habits.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
import numpy as np, matplotlib.pyplot as plt

vec = model.named_steps['tfidfvectorizer']
clf = model.named_steps['logisticregression']
names = np.array(vec.get_feature_names_out())
coef = clf.coef_[0]

top_pos = np.argsort(coef)[-15:]      # towards clf.classes_[1]
top_neg = np.argsort(coef)[:15]       # towards clf.classes_[0]
idx = np.concatenate([top_neg, top_pos])

plt.figure(figsize=(8, 8))
plt.barh(range(len(idx)), coef[idx],
         color=['#8E2436'] * 15 + ['#12706A'] * 15)
plt.yticks(range(len(idx)), names[idx], fontsize=9)
plt.axvline(0, color='black', lw=0.8)
plt.title(f'← towards {clf.classes_[0]}        towards {clf.classes_[1]} →')
plt.tight_layout(); plt.show()

### 6 · How much data did it actually need?

**Copy this into the AI box** — click the word **generate** in the empty cell below, then paste this in:

```text
Retrain the same kind of model on only the first 20 training letters,
then on 50, then on 100, then on 150, then on all of them.
Plot test accuracy against the number of training examples,
and draw the do-nothing baseline as a horizontal line.
```

> This curve is how you answer 'do I need to label more data?' — a question that costs real money in real projects.

Read the shape, not the individual points — each carries a few percentage points of noise, so a dip in the middle means nothing. On this corpus the curve is **still climbing at the right-hand edge**, and that is the useful answer: it says more labelled letters would still buy you accuracy. A curve that had gone flat would have said the opposite — stop labelling, you have enough.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
sizes = [20, 50, 100, 150, len(X_train)]
scores = []
for n in sizes:
    m = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)),
                      LogisticRegression(max_iter=2000))
    m.fit(X_train[:n], y_train[:n])
    scores.append(accuracy_score(y_test, m.predict(X_test)))
    print(f'{n:4d} examples -> {scores[-1]:.1%}')

plt.figure(figsize=(7, 4))
plt.plot(sizes, scores, 'o-', color='#8E2436', label='model')
plt.axhline(base, ls='--', color='grey', label='do-nothing baseline')
plt.xlabel('training examples'); plt.ylabel('test accuracy')
plt.legend(); plt.tight_layout(); plt.show()

---
## 7 · Watch it overfit

The last prompt varied **how much data** the model got. This one varies **how much freedom the model has** to fit that data — and it produces the single most important picture in applied machine learning.

`C` is the setting that controls it. A small `C` keeps the model rigid: it is not allowed to care too much about any one word. A large `C` lets it do whatever it takes to get the training letters right.

Predict, before you run it, what each of the two curves will do.

### 7 · Train and test accuracy, as the model is given more freedom

**Copy this into the AI box** — click the word **generate** in the empty cell below, then paste this in:

```text
Download https://fabsilvestri.github.io/ai-from-the-inside/corpus.zip
and unzip it into the current folder, so that letters.csv and the
archive/ and pdfs/ folders are alongside this notebook.

Train the same TF-IDF + logistic regression classifier at each of these
values of C: 0.01, 0.1, 1, 10, 100, 1000, 10000.

For each one record BOTH the accuracy on the training set and the accuracy
on the test set. Print a table with the two numbers and the gap between
them, then plot both curves against C with a logarithmic x-axis, and draw
the do-nothing baseline as a horizontal line.
Print which C gave the best TEST accuracy.
```

> **Look at the gap, not the level.** Training accuracy climbs to 100% and stays there. Test accuracy stops improving somewhere in the middle and then drifts *down*. Everything to the right of that peak is the model memorising — including memorising the roughly one-in-eight labels that are deliberately wrong.

This is why reporting a single accuracy figure is not enough, and why a model that scores 100% on its training data is a warning rather than an achievement.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
Cs = [0.01, 0.1, 1, 10, 100, 1000, 10000]
tr_scores, te_scores = [], []

print(f"{'C':>8} {'train':>9} {'test':>9} {'gap':>9}")
for C in Cs:
    m = make_pipeline(TfidfVectorizer(), LogisticRegression(C=C, max_iter=5000))
    m.fit(X_train, y_train)
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test, m.predict(X_test))
    tr_scores.append(tr); te_scores.append(te)
    print(f'{C:>8} {tr:>9.1%} {te:>9.1%} {tr - te:>+9.1%}')

best = Cs[int(np.argmax(te_scores))]
print()
print(f'Best test accuracy at C={best}: {max(te_scores):.1%}')
print(f'At C={Cs[-1]} the model scores {tr_scores[-1]:.1%} on data it has seen')
print(f'and {te_scores[-1]:.1%} on data it has not. That gap is the overfitting.')

plt.figure(figsize=(7.5, 4.2))
plt.semilogx(Cs, tr_scores, 'o-', color='#8E2436', label='training data (has seen)')
plt.semilogx(Cs, te_scores, 'o-', color='#12706A', label='test data (has never seen)')
plt.fill_between(Cs, te_scores, tr_scores, color='#B3283C', alpha=0.10)
plt.axhline(base, ls='--', color='grey', label='do-nothing baseline')
plt.xlabel('C  \u2014  how much freedom the model is allowed \u2192')
plt.ylabel('accuracy')
plt.title('The shaded gap is the overfitting')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

---
## 8 · The twist: prove that it memorises

This is the important one. We are going to destroy the relationship between the letters and their labels — by shuffling the labels into random order — and train the very same model on that nonsense.

(If your own split produced a different number of training letters than the reference, that is fine — the shape of the result is what matters, not the exact figures.)

Before you run it, predict what will happen to:

- accuracy on the **training** data
- accuracy on the **test** data

### 8 · Train on nonsense

**Copy this into the AI box** — click the word **generate** in the empty cell below, then paste this in:

```text
Randomly shuffle the training labels so they no longer match their letters,
then train a text classifier on that shuffled data. Give this one plenty of
freedom to memorise: use LogisticRegression with C=100.
Repeat the whole thing 20 times with 20 different shuffles, and report the
AVERAGE training accuracy and the AVERAGE test accuracy, with the
do-nothing baseline next to them.
```

> **Why twenty times?** The test set is 160 letters, so a single run still carries a margin of error of about 4 percentage points — one run tells you very little. Averaging twenty runs is the difference between a number and a measurement, and that is itself one of the lessons.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
import numpy as np

train_scores, test_scores = [], []

for seed in range(20):
    shuffled = np.random.default_rng(seed).permutation(y_train.values)
    noise_model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)),
                                LogisticRegression(max_iter=2000, C=100))
    noise_model.fit(X_train, shuffled)
    train_scores.append(accuracy_score(shuffled, noise_model.predict(X_train)))
    test_scores.append(accuracy_score(y_test, noise_model.predict(X_test)))

tr, te = np.mean(train_scores), np.mean(test_scores)

print('Trained on RANDOM labels, averaged over 20 shuffles:')
print(f'   training accuracy : {tr:.1%}   <- it memorised the nonsense, perfectly')
print(f'   test accuracy     : {te:.1%}   <- it learned nothing at all')
print(f'   do-nothing baseline: {base:.1%}')
print()
print(f'   (single runs ranged {min(test_scores):.1%} to {max(test_scores):.1%} —'
      f' which is why we averaged)')
print()
print('For comparison, the honestly-trained model:')
print(f'   training accuracy : {accuracy_score(y_train, model.predict(X_train)):.1%}')
print(f'   test accuracy     : {acc:.1%}')

---
## What you just did

You collected data, trained a model, evaluated it honestly against a baseline, read its mistakes, inspected what it had learned, measured how much data it needed, and demonstrated overfitting from first principles.

You wrote no code.

### Two things worth trying if you have time

1. Ask the AI: *"Show me what happens if I evaluate on the training data instead of the test data."* Watch the number become meaningless.
2. Ask it: *"Test the model on these two sentences: 'The senate condemned the consul.' and 'The consul condemned the senate.'"* — it cannot tell them apart. It never saw word order at all. That is the limitation Module 1 turns to next.

### Playbook lines earned here

- Never accept an accuracy number without the baseline.
- Ask the machine to show what it got wrong, and which features it used.